# Homework 2 - Build and Analyze Your Own Autoencoder

Download/Load Fashion-MNIST dataset

In [1]:
from torchvision import datasets, transforms

In [3]:
transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, ), (0.5, ))
    ])

In [4]:
train_data = datasets.FashionMNIST(root = './data', train=True, download=True, transform = transform)
test_data = datasets.FashionMNIST(root = './data', train=False, download=True, transform = transform)

In [5]:
from torch.utils.data import DataLoader

train_data = DataLoader(train_data, batch_size = 128, shuffle = True)
test_data  = DataLoader(test_data,  batch_size = 128, shuffle = False)

Create Convolutional Neural Network

In [8]:
import torch, torch.nn as nn, torch.nn.functional as F

class Autoencoder(nn.Module):
    def __init__(self, layers = [32, 64, 128] , dims = 32):
        super().__init__()

        ''' Encoder '''
        encoders = []
        
        for l in layers:
            encoders.append(nn.Conv2d(1, l, 3, stride = 2, padding = 1))
            encoders.append(nn.BatchNorm2d(32))
            encoders.append(nn.ReLU(True))
        
        encoders.append(nn.Dropout(0.2))

        self.encoder = nn.Sequential(*encoders)

        self.fc_mu  = nn.Linear(64 * 7 * 7, dims)
        self.fc_dec = nn.Linear(dims, 64 * 7 * 7)

        ''' Dencoder '''
        decoders = []
        reversed_layers = list(reversed(layers))

        for i in range(len(reversed_layers) - 1):
            decoders.append(nn.ConvTranspose2d(reversed_layers[i], reversed_layers[i + 1], kernel_size = 3, stride = 2, padding = 1, output_padding = 1))
            decoders.append(nn.BatchNorm2d(reversed_layers[i + 1]))
            decoders.append(nn.ReLU(True))

        decoders.append(nn.ConvTranspose2d(reversed_layers[-1], 1, kernel_size = 3, stride = 2, padding = 1, output_padding = 1))
        decoders.append(nn.Sigmoid())

        self.decoder = nn.Sequential(*decoders)

    def forward(self, x):
        # Encode
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        x = self.fc_mu(x)

        # Decode
        x = self.fc_dec(x)
        x = x.view(x.size(0), 64, 7, 7)
        x = self.decoder(x)

        return x

Initialize

In [12]:
layers = [32, 64]
dimentions = [16, 32, 64, 128]

criteria = nn.MSELoss()

epochs = 10

Training

In [13]:
import matplotlib.pyplot as plt

for dim in dimentions:
    for lay in layers:
        ''' Init '''
        model = Autoencoder(layers = layers, dims = dim)
        optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

        training_loss, val_loss = [], []
        
        ''' Training '''
        for epoch in range(epochs):
            model.train()

            loss_cum = 0

            for x, _ in train_data:
                x = x.to()
                recon = model(x)
                loss = criteria(recon, x)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                loss_cum += loss.item()

            training_loss.append(loss_cum / len(train_data))

            model.eval()

            val_cum = 0

            with torch.no_grad():
                for x, _ in test_data:
                    x = x.to()
                    val_cum += criteria(model(x), x).item()

            val_loss.append(val_cum / len(test_data))

            print(f'Dim: {dim} - Epoch {epoch + 1}/{epochs} - Train:{(loss_cum / len(train_data)):4f} - Val:{(val_cum / len(test_data)):4f}')

        ''' Plotting '''
        plt.plot(training_loss, label = 'train')
        plt.plot(val_loss, label = 'val')
        plt.xlabel('Epochs')
        plt.ylabel('MSE Loss')
        plt.legend()
        plt.title(f'Autoencoder Training Curves (neurons: {dim})')
        plt.savefig(f'curve-{dim}.png')
        plt.show()

        ''' Reconstruction '''
        model.eval()
        x, _ = next(iter(test_data))
        x = x[:8].to()
        with torch.no_grad():
            recons = model(x)
        fig, ax = plt.subplots(2, 8, figsize=(8*1.2, 3))
        for i in range(8):
            ax[0,i].imshow(x[i].cpu().squeeze(), cmap='gray'); ax[0,i].axis('off')
            ax[1,i].imshow(recons[i].cpu().squeeze(), cmap='gray'); ax[1,i].axis('off')
        ax[0,0].set_title('Originals'); ax[1,0].set_title('Reconstructions')
        plt.savefig(f'comparation-{dim}.png')
        plt.show()

RuntimeError: Given groups=1, weight of size [64, 1, 3, 3], expected input[128, 32, 14, 14] to have 1 channels, but got 32 channels instead